In [ ]:
import torch
import pandas as pd
import regex as re
from collections import Counter
import numpy as np

In [ ]:
data = pd.read_json("/content/drive/MyDrive/Sports_and_Outdoors_5.json",lines=True)
data.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,AIXZKN4ACSKI,1881509818,David Briner,"[0, 0]",This came in on time and I am veru happy with ...,5,Woks very good,1390694400,"01 26, 2014"
1,A1L5P841VIO02V,1881509818,Jason A. Kramer,"[1, 1]",I had a factory Glock tool that I was using fo...,5,Works as well as the factory tool,1328140800,"02 2, 2012"
2,AB2W04NI4OEAD,1881509818,J. Fernald,"[2, 2]",If you don't have a 3/32 punch or would like t...,4,"It's a punch, that's all.",1330387200,"02 28, 2012"
3,A148SVSWKTJKU6,1881509818,"Jusitn A. Watts ""Maverick9614""","[0, 0]",This works no better than any 3/32 punch you w...,4,It's a punch with a Glock logo.,1328400000,"02 5, 2012"
4,AAAWJ6LW9WMOO,1881509818,Material Man,"[0, 0]",I purchased this thinking maybe I need a speci...,4,"Ok,tool does what a regular punch does.",1366675200,"04 23, 2013"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def preprocess(text):
  return re.sub(r'[^a-z]'," ",text.lower()).split()

In [ ]:
data['reviewText'] = data['reviewText'].apply(lambda x: preprocess(x))

In [ ]:
data['reviewText'].loc[0]

['this',
 'came',
 'in',
 'on',
 'time',
 'and',
 'i',
 'am',
 'veru',
 'happy',
 'with',
 'it',
 'i',
 'haved',
 'used',
 'it',
 'already',
 'and',
 'it',
 'makes',
 'taking',
 'out',
 'the',
 'pins',
 'in',
 'my',
 'glock',
 'very',
 'easy']

In [ ]:
# Next step is to Build Vocabulary and we use collections module but we need to flatten all lists first

In [ ]:
def flatten(dat):
  a = []
  for i in dat:
    a.extend(i)
  return a

flattened_tokens = flatten(data['reviewText'].to_list())

In [ ]:
len(flattened_tokens)

26305309

In [ ]:
word_freq = Counter(flattened_tokens)

In [ ]:
def filter(min_count,word_freq):
  dict = {}
  for i in word_freq.items():
    if i[1] >= min_count:
      dict[i[0]] = i[1]
  return dict

In [ ]:
filtered_word_freq = filter(5,word_freq)
print(len(word_freq))
print(len(filtered_word_freq))

109352
31603


In [ ]:
def build_vocab(wordfrequency):
  vocab = {}
  vocab['PAD'] = 0
  vocab['UNK'] = 1
  for i,j in enumerate(sorted(wordfrequency.items(), key = lambda x : x[1] , reverse=True ) ):
    vocab[j[0]] = i+2
  return vocab

vocabulary = build_vocab(filtered_word_freq)

In [ ]:
def sentence_to_num(data,vocab):
  super_list = []
  for i in data.tolist():
    minor_list = []
    for j in i:
      if j in vocab.keys():
        minor_list.append(vocab[j])
      else:
        minor_list.append(vocab['UNK'])
    super_list.append(minor_list)
  return super_list

In [ ]:
encoded_sentences_notpadded = sentence_to_num(data['reviewText'],vocabulary)

In [ ]:
def max_len(sen):
  t = 0
  for i,j in enumerate(sen):
    if len(j)>t:
      t = len(j)
  return t,i+1

In [ ]:
max,num_sentences = max_len(encoded_sentences_notpadded)
print("Maximum length of Sentence  ",max,"  Number of sentences  ",num_sentences)

Maximum length of Sentence   5773   Number of sentences   296337


In [ ]:
# Using Numpy array to lower the RAM Usage
def padding(sen,maxlen):

  temp = np.zeros((num_sentences,maxlen),dtype = 'int32' )
  for i,j in enumerate(sen):
    if len(j) < maxlen:
      temp[i,:len(j)] = j
  return temp

In [ ]:
padded_sentence = padding(encoded_sentences_notpadded,200)  #Custom Max_len to reduce RAM Usage

In [ ]:
padded_sentence.shape

(296337, 200)

In [ ]:
label = data['overall']
data['overall'].unique()

array([5, 4, 3, 2, 1])

In [ ]:
# We have to first split the data , and use TensorDataset which samples the data as x,y pairs(useful for iterating and batching,shuffling)

index = np.arange(0,num_sentences)
np.random.shuffle(index)  #Shuffling the indices before splitting

from torch.utils.data import TensorDataset,DataLoader

train_data = TensorDataset(torch.tensor(padded_sentence[index[0:int(.7*num_sentences)]]) , torch.tensor(label.iloc[index[0:int(.7*num_sentences)]].to_numpy()))
valid_data = TensorDataset(torch.tensor(padded_sentence[index[int(.7*num_sentences):int(.85*num_sentences)]]) , torch.tensor(label.iloc[index[int(.7*num_sentences):int(.85*num_sentences)]].to_numpy()))
test_data = TensorDataset(torch.tensor(padded_sentence[index[int(.85*num_sentences):num_sentences]]) , torch.tensor(label.iloc[index[int(.85*num_sentences):num_sentences]].to_numpy()))



In [ ]:
# Now , we have to shuffle the data and create batches(similar to batch_size in tensorflow) using Dataloader

train_loader = DataLoader(train_data , shuffle = True , batch_size= 60)
valid_loader = DataLoader(valid_data , shuffle = True , batch_size= 60)
test_loader = DataLoader(test_data , shuffle = True , batch_size= 60)



In [ ]:
class Model(torch.nn.Module):
  def __init__(self, vocab_size , embedding_dim , output_dim) -> None:
    super().__init__()
    self.embedding = torch.nn.Embedding(vocab_size , embedding_dim , padding_idx=0) # O/P is (batches,words_in_each_sen,embedding_dim)
    self.dropout = torch.nn.Dropout(0.4)
    self.forward_layer = torch.nn.Linear(2*embedding_dim,output_dim)

  def forward(self,x):
    embed = self.embedding(x)
    avg_pooling = embed.mean(dim = 1)    #We want to average along the words in each sentence(average pooling) O/P : (batches,embedding_dim)
    max_pooling = embed.max(dim = 1)[0]
    overall_pooling  = torch.cat((avg_pooling,max_pooling),dim = 1)  # concatenating both to get (batch_size, 2*embedding_dim)
    dropout = self.dropout(overall_pooling)
    output = self.forward_layer(dropout)  # Output is 1 to 5 rating
    return output


In [ ]:
weight = 1.0/torch.tensor(data['overall'].value_counts().to_list())
weight = weight/weight.sum()  # Since 5 star rating examples are much more than 1 star , hence penalizing lower ratings more
weight = weight.to('cuda')

model = Model(len(vocabulary),embedding_dim = 100 , output_dim = 5 )
criterion = torch.nn.CrossEntropyLoss(weight = weight)
optimizer = torch.optim.Adam(model.parameters(),lr = 0.0001)
model = model.to('cuda')  # Move model to GPU

print(data['overall'].value_counts())

overall
5    188208
4     64809
3     24071
2     10204
1      9045
Name: count, dtype: int64


In [ ]:
model.train()



def training(epochs):
  for epoch in range(epochs):

    epoch_loss = 0
    correct_pred_train = 0
    correct_pred_val = 0
    val_loss = 0

    for i,(x,y) in enumerate(train_loader):
      x,y = x.to('cuda'),y.to('cuda')
      optimizer.zero_grad()
      predictions = model(x)
      y = y-1
      loss = criterion(predictions,y)
      epoch_loss += loss.item() # converting each loss to number
      loss.backward()   # Doing gradient of loss
      optimizer.step()  # Changing parameters based on derivative
      correct_pred_train += (torch.max(predictions,1)[1] == y).sum().item()   # summing total correct and converting to no

      #torch.max(predictions,1) give (no_of_sentences,classes) and we retrieve highest logit,class_label tuple , but we r interested in label

    train_accuracy = correct_pred_train/len(train_loader.dataset)
    model.eval()  #Set to evaluation mode
    with torch.no_grad():
      for val_x,val_y in (valid_loader):
        val_x,val_y = val_x.to('cuda'),val_y.to('cuda')
        val_y = val_y - 1
        predictions = model(val_x)
        loss = criterion(predictions,val_y)
        val_loss += loss.item()
        correct_pred_val += (torch.max(predictions,1)[1] == val_y).sum().item()

    val_accuracy = correct_pred_val/len(valid_loader.dataset)

    print(f"Epoch:{epoch+1}/{epochs} Training Loss : {epoch_loss} Accuracy : {train_accuracy} Validation Loss : {val_loss} Val accuracy : {val_accuracy}")

training(40)

Epoch:1/40 Training Loss : 2857.8988478779793 Accuracy : 0.6225853881938921 Validation Loss : 569.7298128306866 Val accuracy : 0.6295021484331061
Epoch:2/40 Training Loss : 2538.20424708724 Accuracy : 0.6366379829826211 Validation Loss : 535.5938784778118 Val accuracy : 0.631121909518346
Epoch:3/40 Training Loss : 2413.2872709035873 Accuracy : 0.6399257598765878 Validation Loss : 514.2681132555008 Val accuracy : 0.6347888686418753
Epoch:4/40 Training Loss : 2327.0931867063046 Accuracy : 0.6450068696218092 Validation Loss : 498.6072363257408 Val accuracy : 0.6424602371150255
Epoch:5/40 Training Loss : 2266.273484289646 Accuracy : 0.6491913129414033 Validation Loss : 488.6671356856823 Val accuracy : 0.643652561247216
Epoch:6/40 Training Loss : 2222.3074680268764 Accuracy : 0.6528743943886036 Validation Loss : 482.330731600523 Val accuracy : 0.6499066387707814
Epoch:7/40 Training Loss : 2187.246906787157 Accuracy : 0.6557475835803986 Validation Loss : 474.44368517398834 Val accuracy : 0.6

In [ ]:
print("TOTAL unique words :",len(filtered_word_freq), "\t Embedding matrix Shape:  ", model.embedding.weight.data.shape)

# Extra 2 words are PAD and UNK

TOTAL unique words : 31603 	 Embedding matrix Shape:   torch.Size([31605, 100])


In [ ]:
embeddings = model.embedding.weight.data

In [ ]:
# Try these pairs
sim_good = torch.cosine_similarity(embeddings[vocabulary['great']], embeddings[vocabulary['excellent']], dim=0)
sim_bad = torch.cosine_similarity(embeddings[vocabulary['terrible']], embeddings[vocabulary['horrible']], dim=0)
sim_mixed = torch.cosine_similarity(embeddings[vocabulary['great']], embeddings[vocabulary['terrible']], dim=0)

print(f"Positive Similarity: {sim_good.item()}")
print(f"Negative Similarity: {sim_bad.item()}")
print(f"Opposite Similarity: {sim_mixed.item()}")

Positive Similarity: 0.5177975296974182
Negative Similarity: 0.3528140187263489
Opposite Similarity: -0.35707947611808777


In [ ]:
from torch._C import device
from torch.nn.utils.rnn import pack_padded_sequence , pad_packed_sequence
class Model2(torch.nn.Module):
  def __init__(self, vocab_size , embedding_dim ,hidden_dim ,num_layers, output_dim ) -> None:
    super().__init__()
    self.embedding = torch.nn.Embedding(vocab_size , embedding_dim , padding_idx=0) # O/P is (batches,words_in_each_sen,embedding_dim)
    self.dropout = torch.nn.Dropout(0.4)
    self.forward_layer1 = torch.nn.Linear(2*hidden_dim,output_dim)
    self.lstm = torch.nn.LSTM(embedding_dim,hidden_dim ,num_layers = num_layers, batch_first = True) # O/P (batch,seq,features)
  def forward(self,x,length):
    embed = self.embedding(x)
    padded_sequences = pack_padded_sequence(embed,length,batch_first = True,enforce_sorted = False)
    pad , (h_n,c_n) = self.lstm(padded_sequences)
    out , len = pad_packed_sequence(pad,batch_first = True)   # out is (batch,max_len(200),hidden_dim) and length is just 1D array of seq lengths

# We need to mask the padding now( we can't use general length Tensor having len of each sentence because these are in shuffled batches and need corresponding values.)
    len = len.to('cuda')
    mask = torch.arange(out.size(1),device = 'cuda') < len[:,None] # 200 is the length of each sentence with padding , len[:,None] increases dimension by 1

# torch.sum(out * mask[:,:,None] , dim = 1) / len[:,None]  #numerator is (batch,features) and denominator is (1,batch) , through broadcasting it works and considering
# length of unpadded portion , so torch.mean is useless here

    # torch.where(mask == 0 , -torch.inf , out)  # np.where(condition, value_if_true, value_if_false)

    """combined_pool = torch.cat([
        torch.sum(out * mask[:,:,None] , dim = 1) / len[:,None].float(),
        torch.max(torch.where(mask[:,:,None] ,out,torch.tensor(-float('inf'),device = 'cuda')),dim = 1)[0]],dim = 1) """
    # For every iteration,torch.where creates temporary copy causing very slow training , better approach

    out_clone = out.clone()
    out_clone[mask == 0] = - float('inf')

    combined_pool = torch.cat([
        torch.sum(out * mask[:,:,None] , dim = 1) / len[:,None].float(),
        torch.max(out_clone,dim = 1)[0]],dim = 1)

    dropout = self.dropout(combined_pool)
    output = self.forward_layer1(dropout)  # Output is 1 to 5 rating

    return output


In [ ]:
weight = 1.0/torch.tensor(data['overall'].value_counts().to_list())
weight = (weight/weight.sum())**2.0  # Since 5 star rating examples are much more than 1 star , hence penalizing lower ratings more
weight = weight.to('cuda')

model2 = Model2(len(vocabulary),embedding_dim = 100 ,hidden_dim = 128,num_layers=1, output_dim = 5 )
criterion = torch.nn.CrossEntropyLoss(weight = weight)
optimizer = torch.optim.Adam(model2.parameters(),lr = 0.001,weight_decay=1e-5)
model2 = model2.to('cuda')  # Move model to GPU

In [ ]:
def training(epochs):
  for epoch in range(epochs):
    model2.train()
    epoch_loss = 0
    correct_pred_train = 0
    correct_pred_val = 0
    val_loss = 0

    for i,(x,y) in enumerate(train_loader):
      x,y = x.to('cuda'),y.to('cuda')
      length = (x!=0).sum(dim=1).cpu()   #sentences length are variable in each batch since they are shuffled too , hence runtime calculation req
      length[length==0] = 1
      length.cpu()
      optimizer.zero_grad()
      predictions = model2(x,length)
      y = y-1
      loss = criterion(predictions,y)
      epoch_loss += loss.item() # converting each loss to number
      loss.backward()   # Doing gradient of loss
      optimizer.step()  # Changing parameters based on derivative
      correct_pred_train += (torch.max(predictions,1)[1] == y).sum().item()   # summing total correct and converting to no

      #print(torch.bincount(torch.max(predictions,1)[1]))

      #torch.max(predictions,1) give (no_of_sentences,classes) and we retrieve highest logit,class_label tuple , but we r interested in label

    train_accuracy = correct_pred_train/len(train_loader.dataset)
    model2.eval()  #Set to evaluation mode
    with torch.no_grad():
      for val_x,val_y in (valid_loader):
        val_x,val_y = val_x.to('cuda'),val_y.to('cuda')
        length = (val_x!=0).sum(dim=1).cpu()
        length[length==0] = 1
        length.cpu()
        val_y = val_y - 1
        predictions = model2(val_x,length)
        loss = criterion(predictions,val_y)
        val_loss += loss.item()
        correct_pred_val += (torch.max(predictions,1)[1] == val_y).sum().item()

    val_accuracy = correct_pred_val/len(valid_loader.dataset)

    print(f"Epoch:{epoch+1}/{epochs} Training Loss : {epoch_loss} Accuracy : {train_accuracy} Validation Loss : {val_loss} Val accuracy : {val_accuracy}")

training(40)

Epoch:1/40 Training Loss : 1823.0610593259335 Accuracy : 0.6561428881336322 Validation Loss : 368.5416431725025 Val accuracy : 0.6583203977413331
Epoch:2/40 Training Loss : 1679.7835917770863 Accuracy : 0.671000554390532 Validation Loss : 358.16055607795715 Val accuracy : 0.6674990438910261
Epoch:3/40 Training Loss : 1635.697403088212 Accuracy : 0.6774459469231325 Validation Loss : 354.53642106056213 Val accuracy : 0.6706935726980271
Epoch:4/40 Training Loss : 1599.425193861127 Accuracy : 0.6835008556897342 Validation Loss : 353.1011811643839 Val accuracy : 0.6727407707363164
Epoch:5/40 Training Loss : 1564.2409136146307 Accuracy : 0.6894256032010028 Validation Loss : 352.81793281435966 Val accuracy : 0.6724258171919641
Epoch:6/40 Training Loss : 1528.8612942546606 Accuracy : 0.6947959601802974 Validation Loss : 353.31037859618664 Val accuracy : 0.6765652066320218
Epoch:7/40 Training Loss : 1483.776082456112 Accuracy : 0.7023308506279076 Validation Loss : 367.8413425087929 Val accuracy

KeyboardInterrupt: 

# **We will now train a bidirectional LSTM using Embeddings from a Gensim Word2Vec Model**

In [ ]:
!pip install gensim
!pip install cython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 82.4 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#model = Word2Vec(data['reviewText'].to_list(),vector_size = 200 , sg = 1 , workers = 4 ,window = 5 ,min_count = 5 , epochs = 5)
model = Word2Vec.load("/content/drive/MyDrive/word2vec_sports.model")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
model.wv.most_similar('shoe')

[('shoes', 0.7053035497665405),
 ('sneaker', 0.6419639587402344),
 ('boots', 0.6174113750457764),
 ('toe', 0.6129279136657715),
 ('eeee', 0.5821977257728577),
 ('sneakers', 0.5694416165351868),
 ('laces', 0.568841814994812),
 ('cleats', 0.5591488480567932),
 ('soled', 0.5561547875404358),
 ('vibrams', 0.5550781488418579)]

In [ ]:
# making the embedding matrix

embedded_matrix = torch.zeros((len(vocabulary),model.vector_size),dtype = torch.float32) # Embedding matrix is word and it's corresponding vector

for i in vocabulary:
  if i in model.wv:
    a = torch.from_numpy(model.wv[i].copy())
    embedded_matrix[vocabulary[i]] = a

# Building the actual model

In [ ]:
from torch._C import device
from torch.nn.utils.rnn import pack_padded_sequence , pad_packed_sequence
class Model2(torch.nn.Module):
  def __init__(self, vocab_size , embedding_dim ,hidden_dim ,num_layers, output_dim ,embedding_matrix) -> None:
    super().__init__()
    self.embedding = torch.nn.Embedding(vocab_size , embedding_dim , padding_idx=0) # O/P is (batches,words_in_each_sen,embedding_dim)
    self.embedding.weight.data.copy_(embedding_matrix)
    self.embedding.weight.requires_grad = False # Already checked with False , val_accuracy reached 68.5 plateau and with True 67%
    self.dropout = torch.nn.Dropout(0.4)
    self.forward_layer1 = torch.nn.Linear(4*hidden_dim,output_dim)
    self.num_layers = num_layers
    self.hidden_dim = hidden_dim
    self.lstm = torch.nn.LSTM(embedding_dim,hidden_dim ,num_layers = num_layers, batch_first = True,bidirectional=True) # O/P (batch,seq,2*features or hidden_dim)
  def forward(self,x,length):
    embed = self.embedding(x)
    padded_sequences = pack_padded_sequence(embed,length,batch_first = True,enforce_sorted = False)
    pad , (h_n,c_n) = self.lstm(padded_sequences)
    out , len = pad_packed_sequence(pad,batch_first = True)

    # h_n shape: (num_layers * num_directions, batch, hidden_dim) directions is 2 for past and future

    # h_n = h_n.view(self.num_layers, 2, x.size(0) , self.hidden_dim)   # (1 layer, 2 directions, batch, hidden_dim)

# INCLUDING THE POOLING

    len = len.to('cuda')
    mask = torch.arange(out.size(1),device = 'cuda') < len[:,None]

    out_clone = out.clone()
    out_clone[mask == 0] = - float('inf')

# out has shape (batch,seq_len(time steps),2*hidden_dim) and pooling collapses axis=1 , hence (batch,2*hidden_dim) for both mean and max pool , hence 4*hidden_dim

    combined_pool = torch.cat([
        torch.sum(out * mask[:,:,None] , dim = 1) / len[:,None].float(),
        torch.max(out_clone,dim = 1)[0]],dim = 1)
    """ combined = torch.cat([
            h_n[0,0,:,:],
            h_n[0,1,:,:]],dim = 1) # Since num_layers = 1 , so h_n[-1] = h_n[0])"""

    dropout = self.dropout(combined_pool)
    output = self.forward_layer1(dropout)  # Output is 1 to 5 rating

    return output


In [ ]:
weight = 1.0/torch.tensor(data['overall'].value_counts().to_list())
weight = (weight/weight.sum())**1.5  # Since 5 star rating examples are much more than 1 star , hence penalizing lower ratings more
weight = weight.to('cuda')

model2 = Model2(len(vocabulary),embedding_dim = 200 ,hidden_dim = 128,num_layers=1, output_dim = 5 ,embedding_matrix = embedded_matrix)
criterion = torch.nn.CrossEntropyLoss(weight = weight,label_smoothing = 0.1)
optimizer = torch.optim.Adam(model2.parameters(),lr = 0.001, weight_decay = 1e-5)
model2 = model2.to('cuda')  # Move model to GPU

In [ ]:
def training(epochs):
  for epoch in range(epochs):
    model2.train()
    epoch_loss = 0
    correct_pred_train = 0
    correct_pred_val = 0
    val_loss = 0

    for i,(x,y) in enumerate(train_loader):
      x,y = x.to('cuda'),y.to('cuda')
      length = (x!=0).sum(dim=1).cpu()   #sentences length are variable in each batch since they are shuffled too , hence runtime calculation req
      length[length==0] = 1
      length.cpu()
      optimizer.zero_grad()
      predictions = model2(x,length)
      y = y-1
      loss = criterion(predictions,y)
      epoch_loss += loss.item() # converting each loss to number
      loss.backward()   # Doing gradient of loss
      optimizer.step()  # Changing parameters based on derivative
      correct_pred_train += (torch.max(predictions,1)[1] == y).sum().item()   # summing total correct and converting to no

      #print(torch.bincount(torch.max(predictions,1)[1]))

      #torch.max(predictions,1) give (no_of_sentences,classes) and we retrieve highest logit,class_label tuple , but we r interested in label

    train_accuracy = correct_pred_train/len(train_loader.dataset)
    model2.eval()  #Set to evaluation mode
    with torch.no_grad():
      for val_x,val_y in (valid_loader):
        val_x,val_y = val_x.to('cuda'),val_y.to('cuda')
        length = (val_x!=0).sum(dim=1).cpu()
        length[length==0] = 1
        length.cpu()
        val_y = val_y - 1
        predictions = model2(val_x,length)
        loss = criterion(predictions,val_y)
        val_loss += loss.item()
        correct_pred_val += (torch.max(predictions,1)[1] == val_y).sum().item()

    val_accuracy = correct_pred_val/len(valid_loader.dataset)

    print(f"Epoch:{epoch+1}/{epochs} Training Loss : {epoch_loss} Accuracy : {train_accuracy} Validation Loss : {val_loss} Val accuracy : {val_accuracy}")

training(40)

Epoch:1/40 Training Loss : 1991.1713979542255 Accuracy : 0.6594885144744137 Validation Loss : 405.5892724096775 Val accuracy : 0.6670266135744978
Epoch:2/40 Training Loss : 1891.1137573719025 Accuracy : 0.6725576686672934 Validation Loss : 405.2398451566696 Val accuracy : 0.6767226834041978
Epoch:3/40 Training Loss : 1863.1765488684177 Accuracy : 0.6781112155615012 Validation Loss : 396.06240144371986 Val accuracy : 0.6752828957728735
Epoch:4/40 Training Loss : 1845.9793641865253 Accuracy : 0.6805553546894207 Validation Loss : 397.497632086277 Val accuracy : 0.6735956446424153
Epoch:5/40 Training Loss : 1833.991317152977 Accuracy : 0.6834092607322776 Validation Loss : 393.7692171931267 Val accuracy : 0.6863962565521585
Epoch:6/40 Training Loss : 1821.658233165741 Accuracy : 0.6849567334345699 Validation Loss : 392.1616457104683 Val accuracy : 0.6805696159816427
Epoch:7/40 Training Loss : 1810.5365301966667 Accuracy : 0.6876033456263407 Validation Loss : 392.0333941578865 Val accuracy :

KeyboardInterrupt: 

# 🚀 Phase 1: Model Evolution Summary

## From FFNN to BiLSTM + Pooling — The Road to the ~70% Plateau

---

### 🔹 1. Starting Point: Embedding + FFNN

**Architecture:**  
Embedding Layer $\rightarrow$ Flatten / Simple Aggregation $\rightarrow$ Fully Connected (FC) Layers

**Result:**  
Validation Accuracy $\approx$ **66%**

**Insight:**  
The model captures basic word presence but lacks sequence understanding, leading to limited performance on complex reviews.

---

### 🔹 2. LSTM (No Pooling)

**Architecture:**  
Embedding $\rightarrow$ LSTM $\rightarrow$ Last Hidden State $\rightarrow$ Classifier

**Result:**  
Validation Accuracy $\approx$ **66%**

**Insight:**  
Sequential modeling was added, but relying only on the last hidden state loses information from earlier words and provides a weak representation of the full sentence.

---

### 🔹 3. LSTM + Pooling

**Architecture:**  
BiLSTM Output $\rightarrow$ Mean Pooling + Max Pooling (Concatenation) $\rightarrow$ Classifier

**Result:**  
Validation Accuracy $\approx$ **67%**

**Insight:**  
Pooling captures global information (mean) and strong signals (max), showing a slight improvement over the last hidden state alone.

---

### 🔹 4. Pretrained Embeddings + BiLSTM

**Embeddings:**  
Trained using Gensim Word2Vec with a vector size of 200 and the Skip-gram (sg=1) approach.

**Architecture:**  
Pretrained Embedding $\rightarrow$ BiLSTM $\rightarrow$ Classifier

**Result:**  
Validation Accuracy $\approx$ **68%**

**Insight:**  
Pretrained embeddings provide better semantic understanding, while the BiLSTM captures both forward and backward context.

---

### 🔹 5. Pretrained + BiLSTM + Pooling

**Architecture:**  
BiLSTM $\rightarrow$ Mean Pooling + Max Pooling $\rightarrow$ Classifier

**Result:**  
Validation Accuracy $\approx$ **69%**

**Insight:**  
This was the best performance in the 5-class setup, combining contextual understanding (BiLSTM) with global aggregation (pooling).

---

### ⚠️ 6. Key Observation: Performance Plateau

Across all architectural improvements, the model's accuracy moved incrementally:  
$$66\% \rightarrow 67\% \rightarrow 68\% \rightarrow 69\%$$

The model never crossed the **~70%** threshold.

**Why did this plateau happen?**  
- **Class Imbalance:** 5★ dominates the dataset heavily.  
- **Label Ambiguity:** Hard to distinguish 4 vs 5 or 1 vs 2.  
- **Dataset Noise:** Natural language reviews are inconsistent.  
- **Task Difficulty:** Fine-grained sentiment (5 classes) is inherently hard.

---

### 🔹 7. Fine-tuning Pretrained Embeddings (Unfreezing)

**Experiment:**  
Set `embedding.weight.requires_grad = True`

**Result:**  
Validation Accuracy dropped to around **67%**

**Insight:**  
Unfreezing caused overfitting and a loss of pretrained semantic structure. Updates were too noisy due to limited dataset quality and class imbalance.

**Conclusion:**  
Pretrained embeddings worked better when frozen.

---

### 🔥 8. Final Understanding from This Phase

- **Architecture improvements** gave diminishing returns: each step only yielded a small gain (~1%).  
- **Data quality dominates performance:** Model complexity was less impactful than dataset quality.  
- **Pretrained embeddings** help, but fine-tuning is not always beneficial.  
- **Pooling** is a strong baseline: simple yet effective aggregation for sentence representations.  
- **BiLSTM > LSTM:** Context from both directions matters for sentiment.

---

## 🏁 Final Conclusion

Despite trying FFNN, LSTM, BiLSTM, Pooling, and Pretrained embeddings, performance saturated around **~69%** for the 5-class problem.

**The Transition Point:**  
This plateau led to the key realization that the limitation was not just the model, but the problem formulation itself.  
This directly motivated the exploration of **Attention Mechanisms** and the **Rating $\rightarrow$ Sentiment (5-class $\rightarrow$ 3-class)** transformation.